In [1]:
import os
import sys
import gzip
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from limen import (Gene, Transcript, identifiability, build_matrix,
                   incidence_patterns, load_annotation, detect_format)
import limen

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("limen  :", limen.__version__, "from", limen.__file__)

python : 3.11.16
numpy  : 2.4.6
limen  : 0.1.0 from /Users/ib/Code/limen/src/limen/__init__.py


In [2]:
REF  = Path(os.environ.get("LIMEN_REF",  Path.home() / "Code" / "ReferenceGenome"))
WORK = Path(os.environ.get("LIMEN_WORK", Path.home() / "work" / "limen"))
(WORK / "results").mkdir(parents=True, exist_ok=True)

# annotations available on this machine
ANNOT = {
    "Chlamydomonas": REF / "CreinhardtiiCC_4532_707_v6.1.gene.gff3",
    "Arabidopsis"  : REF / "Araport11_GFF3_genes_transposons.201606.std_r_luc.gff3",
    "Oryza"        : REF / "IRGSP-1.0_representative_2020-09-09.rev2.std_r_luc.gff3",
    "Nicotiana"    : REF / "Niben101_annotation.gene_models.fix.std_r_luc.gff3",
}

# the read-length ladder: Illumina through to full-length long reads
READ_LENGTHS = [100, 150, 250, 300, 500, 1000, 2000, 4000, 8000, 16000]

# short-read paired-end grid, for comparison with published short-read work
FRAGMENT_LENGTHS = [200, 300, 400, 600, 900]

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (7, 4)

for name, p in ANNOT.items():
    size = f"{p.stat().st_size/1e6:.0f} MB" if p.exists() else "MISSING"
    print(f"{name:<15} {size:>8}  {p.name}")

Chlamydomonas      56 MB  CreinhardtiiCC_4532_707_v6.1.gene.gff3
Arabidopsis       144 MB  Araport11_GFF3_genes_transposons.201606.std_r_luc.gff3
Oryza              70 MB  IRGSP-1.0_representative_2020-09-09.rev2.std_r_luc.gff3
Nicotiana         144 MB  Niben101_annotation.gene_models.fix.std_r_luc.gff3


In [3]:
path = ANNOT["Chlamydomonas"]

t0 = time.time()
genes, stats = load_annotation(str(path), verbose=True)
print(f"\nparsed in {time.time() - t0:.1f} s")
print(f"format detected: {detect_format(str(path))}")


parsed in 0.7 s
format detected: gff3


[limen] /Users/ib/Code/ReferenceGenome/CreinhardtiiCC_4532_707_v6.1.gene.gff3 (gff3)
  lines read        : 390,271
  feature types     : CDS=274,602, five_prime_UTR=34,130, mRNA=32,672, three_prime_UTR=31,168, gene=16,883, transposable_element_gene=810
  structure source  : exon features 0 tx, CDS/UTR fallback 32,672 tx
  genes             : 16,883
  transcripts       : 31,858
  skipped (seqid)   : 0
  skipped (exclude) : 0
  skipped (TE)      : 810
  orphan features   : 0
  sequences         : chromosome_12(31,842), chromosome_03(30,142), chromosome_02(29,935), chromosome_06(27,394), chromosome_16(26,163), chromosome_01(25,976), chromosome_10(23,236), chromosome_07(20,730)


In [4]:
counts = np.array([g.n_iso for g in genes])

print(f"genes                  : {len(genes):,}")
print(f"distinct transcripts   : {counts.sum():,}")
print(f"mean isoforms per gene : {counts.mean():.2f}")
print(f"genes with 2+ isoforms : {(counts >= 2).sum():,} ({100*(counts>=2).mean():.1f}%)")
print(f"max isoforms in a gene : {counts.max()}")
print()
for k in range(1, 11):
    n = int((counts == k).sum())
    if n:
        print(f"  {k:>2} isoform(s): {n:>6,}  {'#' * int(70*n/len(genes))}")
hi = int((counts > 10).sum())
if hi:
    print(f"  >10 isoforms : {hi:>6,}")

genes                  : 16,883
distinct transcripts   : 31,858
mean isoforms per gene : 1.89
genes with 2+ isoforms : 7,287 (43.2%)
max isoforms in a gene : 21

   1 isoform(s):  9,596  #######################################
   2 isoform(s):  3,793  ###############
   3 isoform(s):  1,713  #######
   4 isoform(s):    817  ###
   5 isoform(s):    416  #
   6 isoform(s):    215  
   7 isoform(s):    134  
   8 isoform(s):     69  
   9 isoform(s):     50  
  10 isoform(s):     31  
  >10 isoforms :     49


In [5]:
exon_len, intron_len, tx_len, n_exons = [], [], [], []
for g in genes:
    for t in g.transcripts:
        tx_len.append(t.length)
        n_exons.append(len(t.exons))
        exon_len.extend(e - s for s, e in t.exons)
        intron_len.extend(s2 - e1 for (_, e1), (s2, _) in zip(t.exons, t.exons[1:]))

arch = pd.DataFrame({
    "metric": ["exon length", "intron length", "transcript length", "exons per transcript"],
    "n":      [len(exon_len), len(intron_len), len(tx_len), len(n_exons)],
    "median": [np.median(x) for x in (exon_len, intron_len, tx_len, n_exons)],
    "mean":   [np.mean(x)   for x in (exon_len, intron_len, tx_len, n_exons)],
    "p10":    [np.percentile(x, 10) for x in (exon_len, intron_len, tx_len, n_exons)],
    "p90":    [np.percentile(x, 90) for x in (exon_len, intron_len, tx_len, n_exons)],
}).round(1)
arch

,metric,n,median,mean,p10,p90
0,exon length,280442,152.0,360.4,63.0,903.0
1,intron length,248584,222.0,265.4,123.0,424.0
2,transcript length,31858,2718.0,3172.4,981.0,5697.6
3,exons per transcript,31858,7.0,8.8,2.0,17.0


In [6]:
multi = sorted((g for g in genes if g.n_iso >= 2), key=lambda g: -g.n_iso)
g = multi[0]

print(f"{g.id}   {g.seqid} {g.strand}   {g.n_iso} distinct structures\n")
for t in g.transcripts[:4]:
    spans = "  ".join(f"{s}-{e}" for s, e in t.exons[:6])
    more = "  ..." if len(t.exons) > 6 else ""
    print(f"  {t.id}")
    print(f"    {len(t.exons)} exons, {t.length} nt")
    print(f"    {spans}{more}\n")

print("verify in a terminal with:")
print(f'  grep "{g.transcripts[0].id}" "$LIMEN_REF/CreinhardtiiCC_4532_707_v6.1.gene.gff3" | cut -f3,4,5')

Cre03.g145127_4532.v6.1   chromosome_03 +   21 distinct structures

  Cre03.g145127_4532.1.v6.1
    22 exons, 14340 nt
    512260-512450  512545-512663  512728-512850  513119-514828  515012-515348  515492-515791  ...

  Cre03.g145127_4532.10.v6.1
    23 exons, 13984 nt
    512260-512450  512545-512663  512728-512850  513119-514828  515012-515348  515492-515791  ...

  Cre03.g145127_4532.11.v6.1
    23 exons, 14057 nt
    512260-512450  512545-512663  512728-512850  513119-514828  515012-515348  515492-515791  ...

  Cre03.g145127_4532.12.v6.1
    23 exons, 13732 nt
    512260-512450  512545-512663  512728-512850  513119-514828  515012-515348  515492-515791  ...

verify in a terminal with:
  grep "Cre03.g145127_4532.1.v6.1" "$LIMEN_REF/CreinhardtiiCC_4532_707_v6.1.gene.gff3" | cut -f3,4,5


In [7]:
sample = multi[:300]
t0 = time.time()

rows = []
for R in (100, 150, 300, 500, 1000, 2000, 5000, 20000):
    res = [identifiability(g, R) for g in sample]
    rows.append({
        "read_length": R,
        "identifiable_%": 100 * np.mean([r.identifiable for r in res]),
        "mean_deficiency": np.mean([r.deficiency for r in res]),
    })

print(f"{len(sample)} genes, {time.time()-t0:.1f} s\n")
pd.DataFrame(rows).round(2)

300 genes, 4.9 s



,read_length,identifiable_%,mean_deficiency
0,100,97.67,0.05
1,150,98.00,0.05
2,300,98.00,0.04
3,500,98.67,0.02
4,1000,99.67,0.00
5,2000,99.67,0.00
6,5000,100.00,0.00
7,20000,100.00,0.00


In [8]:
R = 150
deficient = [(g, identifiability(g, R)) for g in multi[:300]]
deficient = [(g, r) for g, r in deficient if not r.identifiable]

print(f"{len(deficient)} of 300 are not identifiable at {R} bp\n")

for g, r in deficient[:3]:
    print(f"{g.id}  {g.n_iso} isoforms, rank {r.rank}, blind in {r.deficiency}")
    lens = [t.length for t in g.transcripts]
    nex  = [len(t.exons) for t in g.transcripts]
    print(f"   transcript lengths {min(lens)}-{max(lens)} nt, exons {min(nex)}-{max(nex)}")
    M = build_matrix(g, R)
    uniq = int((M.sum(axis=1) == 1).sum())
    print(f"   {M.shape[0]} distinct read patterns, {uniq} of them unique to one isoform\n")

6 of 300 are not identifiable at 150 bp

Cre12.g501752_4532.v6.1  16 isoforms, rank 10, blind in 6
   transcript lengths 1239-2047 nt, exons 7-8
   26 distinct read patterns, 2 of them unique to one isoform

Cre05.g245500_4532.v6.1  15 isoforms, rank 14, blind in 1
   transcript lengths 7435-8006 nt, exons 36-39
   33 distinct read patterns, 10 of them unique to one isoform

Cre13.g586750_4532.v6.1  11 isoforms, rank 10, blind in 1
   transcript lengths 2317-6853 nt, exons 1-17
   21 distinct read patterns, 4 of them unique to one isoform

